# PySpark & la MLlib

## Récupération des données

In [1]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s titanic
!ls -lh titanic/

remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (2/2), 1.94 KiB | 1.94 MiB/s, done.
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 0), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), 33.71 KiB | 33.71 MiB/s, done.
✓ titanic — 5 fichiers, 100K
total 100K
-rw-r--r-- 1 root root  237 Oct  8 21:51 data_description.txt
-rw-r--r-- 1 root root 3.2K Oct  8 21:51 gender_submission.csv
-rw-r--r-- 1 root root  709 Oct  8 21:51 README.md
-rw-r--r-- 1 root root  28K Oct  8 21:51 test.csv
-rw-r--r-- 1 root root  60K Oct  8 21:51 train.csv


## Import des libraries nécessaires

In [2]:
!pip install inflection pyspark

  Using cached inflection-0.5.1-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached inflection-0.5.1-py2.py3-none-any.whl (9.5 kB)


In [3]:
from pathlib import Path

import inflection
import pyspark
import pyspark.ml.feature as MF
import pyspark.sql.functions as F

## Chargement des données

Le fichier csv est chargé dans un `Dataframe` spark.

In [4]:
data_dir = Path("titanic")
csv_path = data_dir / "train.csv"

In [5]:
spark = (pyspark.sql.SparkSession.builder.appName("Démonstration de PySpark")
                                         .getOrCreate())
df = spark.read.csv(str(csv_path), header=True, inferSchema=True)

In [6]:
passengers_count = df.count()

In [7]:
print(passengers_count)

891


In [8]:
df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [9]:
df.describe().show()

+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|summary|      PassengerId|           Survived|            Pclass|                Name|   Sex|               Age|             SibSp|              Parch|            Ticket|             Fare|Cabin|Embarked|
+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|  count|              891|                891|               891|                 891|   891|               714|               891|                891|               891|              891|  204|     889|
|   mean|            446.0| 0.3838383838383838| 2.308641975308642|                NULL|  NULL| 29.69911764705882|0.5230078563411896|0.38159371492704824|260318.54916792738| 32.20420

In [10]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



## Renommage des colonnes pour utiliser la convention Python du snake case

In [11]:
df = df.toDF(*(inflection.underscore(c) for c in df.columns))
df.show(5)

+------------+--------+------+--------------------+------+----+------+-----+----------------+-------+-----+--------+
|passenger_id|survived|pclass|                name|   sex| age|sib_sp|parch|          ticket|   fare|cabin|embarked|
+------------+--------+------+--------------------+------+----+------+-----+----------------+-------+-----+--------+
|           1|       0|     3|Braund, Mr. Owen ...|  male|22.0|     1|    0|       A/5 21171|   7.25| NULL|       S|
|           2|       1|     1|Cumings, Mrs. Joh...|female|38.0|     1|    0|        PC 17599|71.2833|  C85|       C|
|           3|       1|     3|Heikkinen, Miss. ...|female|26.0|     0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|           4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|     1|    0|          113803|   53.1| C123|       S|
|           5|       0|     3|Allen, Mr. Willia...|  male|35.0|     0|    0|          373450|   8.05| NULL|       S|
+------------+--------+------+--------------------+------+----+-

## Exploration des données

In [12]:
df.select("survived", "pclass", "embarked").show()

+--------+------+--------+
|survived|pclass|embarked|
+--------+------+--------+
|       0|     3|       S|
|       1|     1|       C|
|       1|     3|       S|
|       1|     1|       S|
|       0|     3|       S|
|       0|     3|       Q|
|       0|     1|       S|
|       0|     3|       S|
|       1|     3|       S|
|       1|     2|       C|
|       1|     3|       S|
|       1|     1|       S|
|       0|     3|       S|
|       0|     3|       S|
|       0|     3|       S|
|       1|     2|       S|
|       0|     3|       Q|
|       1|     2|       S|
|       0|     3|       S|
|       1|     3|       C|
+--------+------+--------+
only showing top 20 rows


In [13]:
df.groupBy("survived").count().show()

+--------+-----+
|survived|count|
+--------+-----+
|       1|  342|
|       0|  549|
+--------+-----+



In [14]:
df.groupBy("sex", "survived").count().show()

+------+--------+-----+
|   sex|survived|count|
+------+--------+-----+
|  male|       0|  468|
|female|       1|  233|
|female|       0|   81|
|  male|       1|  109|
+------+--------+-----+



In [15]:
df.groupBy("pclass", "survived").count().show()

+------+--------+-----+
|pclass|survived|count|
+------+--------+-----+
|     1|       0|   80|
|     3|       1|  119|
|     1|       1|  136|
|     2|       1|   87|
|     2|       0|   97|
|     3|       0|  372|
+------+--------+-----+



## Préparation des données

### Valeurs nulles

In [16]:
def null_value_count(df: pyspark.sql.DataFrame):
  null_columns_counts = []
  for k in df.columns:
    null_rows = df.where(F.col(k).isNull()).count()
    if null_rows:
      null_columns_counts.append((k, null_rows))
  return null_columns_counts

null_columns_count_list = null_value_count(df)
spark.createDataFrame(null_columns_count_list,
                      ['column_with_nulls', 'nulls_count']).show()

+-----------------+-----------+
|column_with_nulls|nulls_count|
+-----------------+-----------+
|              age|        177|
|            cabin|        687|
|         embarked|          2|
+-----------------+-----------+



### Analyse de l'âge des passagers

In [17]:
mean_age = df.select(F.mean('age')).collect()[0][0]
print(mean_age)

29.69911764705882


On pourrait utiliser cet âge moyen pour l'imputation des âges inconnus, mais nous allons voir comment aller un peu plus loin et calculer âge moyen par civilité.

In [18]:
df = df.withColumn("title", F.regexp_extract(F.col("name"), r"([A-Za-z]+)\.", 1))
df.show()

+------------+--------+------+--------------------+------+----+------+-----+----------------+-------+-----+--------+------+
|passenger_id|survived|pclass|                name|   sex| age|sib_sp|parch|          ticket|   fare|cabin|embarked| title|
+------------+--------+------+--------------------+------+----+------+-----+----------------+-------+-----+--------+------+
|           1|       0|     3|Braund, Mr. Owen ...|  male|22.0|     1|    0|       A/5 21171|   7.25| NULL|       S|    Mr|
|           2|       1|     1|Cumings, Mrs. Joh...|female|38.0|     1|    0|        PC 17599|71.2833|  C85|       C|   Mrs|
|           3|       1|     3|Heikkinen, Miss. ...|female|26.0|     0|    0|STON/O2. 3101282|  7.925| NULL|       S|  Miss|
|           4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|     1|    0|          113803|   53.1| C123|       S|   Mrs|
|           5|       0|     3|Allen, Mr. Willia...|  male|35.0|     0|    0|          373450|   8.05| NULL|       S|    Mr|
|       

In [19]:
df.select("title").distinct().show()

+--------+
|   title|
+--------+
|     Don|
|    Miss|
|Countess|
|     Col|
|     Rev|
|    Lady|
|  Master|
|     Mme|
|    Capt|
|      Mr|
|      Dr|
|     Mrs|
|     Sir|
|Jonkheer|
|    Mlle|
|   Major|
|      Ms|
+--------+



In [20]:
replacements = dict(Miss=["Mlle", "Ms"],
                    Mr=["Dr", "Major", "Capt", "Sir", "Don"],
                    Mrs=["Lady", "Countess", "Mme"],
                    Other=["Jonkheer", "Col", "Rev"])
df = df.replace({name: title
                 for title, names in replacements.items()
                 for name in names},
                subset=["title"])

In [21]:
df.select("title").distinct().show()

+------+
| title|
+------+
|  Miss|
| Other|
|Master|
|    Mr|
|   Mrs|
+------+



In [22]:
average_title_ages = df.groupby("title").agg(F.avg("age").alias("average_age"))
average_title_ages.show()

+------+------------------+
| title|       average_age|
+------+------------------+
|  Miss|21.845637583892618|
| Other|45.888888888888886|
|Master| 4.574166666666667|
|    Mr| 32.73960880195599|
|   Mrs|35.873873873873876|
+------+------------------+



### Imputation de l'âge

On attribue l'âge moyen du groupe de passagers partageant le même titre.

In [23]:
df = (df.join(average_title_ages, "title")
        .withColumn("age", F.when(F.col("age").isNull(),
                                  F.col("average_age"))
                            .otherwise(F.col("age")))
        .drop("average_age"))
df.show()

+------+------------+--------+------+--------------------+------+------------------+------+-----+----------------+-------+-----+--------+
| title|passenger_id|survived|pclass|                name|   sex|               age|sib_sp|parch|          ticket|   fare|cabin|embarked|
+------+------------+--------+------+--------------------+------+------------------+------+-----+----------------+-------+-----+--------+
|    Mr|           1|       0|     3|Braund, Mr. Owen ...|  male|              22.0|     1|    0|       A/5 21171|   7.25| NULL|       S|
|   Mrs|           2|       1|     1|Cumings, Mrs. Joh...|female|              38.0|     1|    0|        PC 17599|71.2833|  C85|       C|
|  Miss|           3|       1|     3|Heikkinen, Miss. ...|female|              26.0|     0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|   Mrs|           4|       1|     1|Futrelle, Mrs. Ja...|female|              35.0|     1|    0|          113803|   53.1| C123|       S|
|    Mr|           5|       0|    

La caractéristique *embarked* (le port d'embarcation) a seulement deux valeurs manquantes. Nous allons imputer le mode pour ces deux valeurs.

In [24]:
df.groupBy("embarked").count().show()

+--------+-----+
|embarked|count|
+--------+-----+
|       Q|   77|
|    NULL|    2|
|       C|  168|
|       S|  644|
+--------+-----+



In [25]:
df = df.na.fill(dict(embarked="S"))

La caractéristique *cabin* peut être supprimée : le numéro de cabine est quasi unique pour chaque passager qui est en cabine et va provoquer rapidement du sur-apprentissage.

In [26]:
df = df.drop("cabin")

In [27]:
df.printSchema()

root
 |-- title: string (nullable = true)
 |-- passenger_id: integer (nullable = true)
 |-- survived: integer (nullable = true)
 |-- pclass: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- age: double (nullable = true)
 |-- sib_sp: integer (nullable = true)
 |-- parch: integer (nullable = true)
 |-- ticket: string (nullable = true)
 |-- fare: double (nullable = true)
 |-- embarked: string (nullable = false)



### Ajout d'une nouvelle caractéristique.

La taille de la famille.

In [28]:
df = df.withColumn("family_size", F.col("sib_sp") + F.col("parch") + 1)

In [29]:
df.groupBy("family_size").count().show()

+-----------+-----+
|family_size|count|
+-----------+-----+
|          1|  537|
|          6|   22|
|          3|  102|
|          5|   15|
|          4|   29|
|          8|    6|
|          7|   12|
|         11|    7|
|          2|  161|
+-----------+-----+



In [30]:
df = df.withColumn("alone", (F.col("family_size") == 1).cast("long"))
df.show()

+------+------------+--------+------+--------------------+------+------------------+------+-----+----------------+-------+--------+-----------+-----+
| title|passenger_id|survived|pclass|                name|   sex|               age|sib_sp|parch|          ticket|   fare|embarked|family_size|alone|
+------+------------+--------+------+--------------------+------+------------------+------+-----+----------------+-------+--------+-----------+-----+
|    Mr|           1|       0|     3|Braund, Mr. Owen ...|  male|              22.0|     1|    0|       A/5 21171|   7.25|       S|          2|    0|
|   Mrs|           2|       1|     1|Cumings, Mrs. Joh...|female|              38.0|     1|    0|        PC 17599|71.2833|       C|          2|    0|
|  Miss|           3|       1|     3|Heikkinen, Miss. ...|female|              26.0|     0|    0|STON/O2. 3101282|  7.925|       S|          1|    1|
|   Mrs|           4|       1|     1|Futrelle, Mrs. Ja...|female|              35.0|     1|    0|   

In [31]:
df.columns

['title',
 'passenger_id',
 'survived',
 'pclass',
 'name',
 'sex',
 'age',
 'sib_sp',
 'parch',
 'ticket',
 'fare',
 'embarked',
 'family_size',
 'alone']

### Encodage de variables catégorielles

In [32]:
indexers = [MF.StringIndexer(inputCol=column,
                             outputCol=f"{column}_index").fit(df)
            for column in ["sex", "embarked", "title"]]
pipeline = pyspark.ml.Pipeline(stages=indexers)
df = pipeline.fit(df).transform(df)
df.show()

+------+------------+--------+------+--------------------+------+------------------+------+-----+----------------+-------+--------+-----------+-----+---------+--------------+-----------+
| title|passenger_id|survived|pclass|                name|   sex|               age|sib_sp|parch|          ticket|   fare|embarked|family_size|alone|sex_index|embarked_index|title_index|
+------+------------+--------+------+--------------------+------+------------------+------+-----+----------------+-------+--------+-----------+-----+---------+--------------+-----------+
|    Mr|           1|       0|     3|Braund, Mr. Owen ...|  male|              22.0|     1|    0|       A/5 21171|   7.25|       S|          2|    0|      0.0|           0.0|        0.0|
|   Mrs|           2|       1|     1|Cumings, Mrs. Joh...|female|              38.0|     1|    0|        PC 17599|71.2833|       C|          2|    0|      1.0|           1.0|        2.0|
|  Miss|           3|       1|     3|Heikkinen, Miss. ...|female|

In [33]:
df.printSchema()

root
 |-- title: string (nullable = true)
 |-- passenger_id: integer (nullable = true)
 |-- survived: integer (nullable = true)
 |-- pclass: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- age: double (nullable = true)
 |-- sib_sp: integer (nullable = true)
 |-- parch: integer (nullable = true)
 |-- ticket: string (nullable = true)
 |-- fare: double (nullable = true)
 |-- embarked: string (nullable = false)
 |-- family_size: integer (nullable = true)
 |-- alone: long (nullable = true)
 |-- sex_index: double (nullable = false)
 |-- embarked_index: double (nullable = false)
 |-- title_index: double (nullable = false)



### Suppression des caractéristiques non numériques

In [34]:
df = df.drop(*(name for name, dtype in df.dtypes if dtype == "string"))

In [35]:
df.show()

+------------+--------+------+------------------+------+-----+-------+-----------+-----+---------+--------------+-----------+
|passenger_id|survived|pclass|               age|sib_sp|parch|   fare|family_size|alone|sex_index|embarked_index|title_index|
+------------+--------+------+------------------+------+-----+-------+-----------+-----+---------+--------------+-----------+
|           1|       0|     3|              22.0|     1|    0|   7.25|          2|    0|      0.0|           0.0|        0.0|
|           2|       1|     1|              38.0|     1|    0|71.2833|          2|    0|      1.0|           1.0|        2.0|
|           3|       1|     3|              26.0|     0|    0|  7.925|          1|    1|      1.0|           0.0|        1.0|
|           4|       1|     1|              35.0|     1|    0|   53.1|          2|    0|      1.0|           0.0|        2.0|
|           5|       0|     3|              35.0|     0|    0|   8.05|          1|    1|      0.0|           0.0|     

### Création des vecteurs de caractéristiques

In [36]:
feature = MF.VectorAssembler(
    inputCols=df.drop("survived", "passenger_id").columns, outputCol="features")
feature_vector = feature.transform(df)

In [37]:
feature_vector.show()

+------------+--------+------+------------------+------+-----+-------+-----------+-----+---------+--------------+-----------+--------------------+
|passenger_id|survived|pclass|               age|sib_sp|parch|   fare|family_size|alone|sex_index|embarked_index|title_index|            features|
+------------+--------+------+------------------+------+-----+-------+-----------+-----+---------+--------------+-----------+--------------------+
|           1|       0|     3|              22.0|     1|    0|   7.25|          2|    0|      0.0|           0.0|        0.0|(10,[0,1,2,4,5],[...|
|           2|       1|     1|              38.0|     1|    0|71.2833|          2|    0|      1.0|           1.0|        2.0|[1.0,38.0,1.0,0.0...|
|           3|       1|     3|              26.0|     0|    0|  7.925|          1|    1|      1.0|           0.0|        1.0|[3.0,26.0,0.0,0.0...|
|           4|       1|     1|              35.0|     1|    0|   53.1|          2|    0|      1.0|           0.0|     

### Séparation en ensembles d'entraînement et de test

In [38]:
training_data, test_data = feature_vector.randomSplit([0.8, 0.2], seed=11)

## Apprentissage de modèles

### Régression logistique

In [39]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(labelCol="survived", featuresCol="features")
lr_model = lr.fit(training_data)
lr_prediction = lr_model.transform(test_data)
lr_prediction.select("prediction", "survived", "features").show()
evaluator = pyspark.ml.evaluation.MulticlassClassificationEvaluator(
    labelCol="survived", predictionCol="prediction", metricName="accuracy")

# Évaluation
lr_accuracy = evaluator.evaluate(lr_prediction)
print(f"Exactitude d'une régression logistique : {lr_accuracy:.2f}")

+----------+--------+--------------------+
|prediction|survived|            features|
+----------+--------+--------------------+
|       0.0|       0|[3.0,32.739608801...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|[3.0,39.0,1.0,5.0...|
|       1.0|       0|[3.0,14.0,0.0,0.0...|
|       0.0|       0|[3.0,2.0,4.0,1.0,...|
|       0.0|       1|(10,[0,1,4,5,6],[...|
|       0.0|       1|(10,[0,1,4,5,6],[...|
|       0.0|       0|[1.0,19.0,3.0,2.0...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[3.0,21.845637583...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[3.0,14.0,1.0,0.0...|
|       1.0|       1|[3.0,21.845637583...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|[3.0,16.0,5.0,2.0...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[2.0,0.83,0.0,2.0...|
|       1.0|       1|[1.0,23.0,3.0,2.0...|
+----------

### Forêt aléatoire

In [40]:
from pyspark.ml.classification import RandomForestClassifier

rfc = RandomForestClassifier(labelCol="survived", featuresCol="features")
rfc_model = rfc.fit(training_data)
rfc_predictions = rfc_model.transform(test_data)
rfc_predictions.select("prediction", "survived", "features").show()

# Évaluation
rfc_accuracy = evaluator.evaluate(rfc_predictions)
print(f"Exactitude d'une forêt aléatoire : {rfc_accuracy:.2f}")

+----------+--------+--------------------+
|prediction|survived|            features|
+----------+--------+--------------------+
|       0.0|       0|[3.0,32.739608801...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|[3.0,39.0,1.0,5.0...|
|       1.0|       0|[3.0,14.0,0.0,0.0...|
|       0.0|       0|[3.0,2.0,4.0,1.0,...|
|       0.0|       1|(10,[0,1,4,5,6],[...|
|       0.0|       1|(10,[0,1,4,5,6],[...|
|       0.0|       0|[1.0,19.0,3.0,2.0...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[3.0,21.845637583...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[3.0,14.0,1.0,0.0...|
|       1.0|       1|[3.0,21.845637583...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|[3.0,16.0,5.0,2.0...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[2.0,0.83,0.0,2.0...|
|       1.0|       1|[1.0,23.0,3.0,2.0...|
+----------

### Arbre boosté

In [41]:
from pyspark.ml.classification import GBTClassifier

gbtc = GBTClassifier(labelCol="survived", featuresCol="features", maxIter=10)
gbtc_model = gbtc.fit(training_data)
gbtc_predictions = gbtc_model.transform(test_data)
gbtc_predictions.select("prediction", "survived", "features").show()

# Évaluation
gbtc_accuracy = evaluator.evaluate(gbtc_predictions)
print(f"Exactitude d'un arbre boosté : {gbtc_accuracy:.2f}")

+----------+--------+--------------------+
|prediction|survived|            features|
+----------+--------+--------------------+
|       0.0|       0|[3.0,32.739608801...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|[3.0,39.0,1.0,5.0...|
|       1.0|       0|[3.0,14.0,0.0,0.0...|
|       0.0|       0|[3.0,2.0,4.0,1.0,...|
|       0.0|       1|(10,[0,1,4,5,6],[...|
|       0.0|       1|(10,[0,1,4,5,6],[...|
|       0.0|       0|[1.0,19.0,3.0,2.0...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[3.0,21.845637583...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[3.0,14.0,1.0,0.0...|
|       1.0|       1|[3.0,21.845637583...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|[3.0,16.0,5.0,2.0...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[2.0,0.83,0.0,2.0...|
|       0.0|       1|[1.0,23.0,3.0,2.0...|
+----------

### Séparateur à vaste marge

In [42]:
from pyspark.ml.classification import LinearSVC

svmc = LinearSVC(labelCol="survived", featuresCol="features")
svmc_model = svmc.fit(training_data)
svmc_predictions = svmc_model.transform(test_data)
svmc_predictions.select("prediction", "survived", "features").show()

# Évaluation
svmc_accuracy = evaluator.evaluate(svmc_predictions)
print(f"Exactitude d'un séparateur à vaste marge : {svmc_accuracy:.2f}")

+----------+--------+--------------------+
|prediction|survived|            features|
+----------+--------+--------------------+
|       0.0|       0|[3.0,32.739608801...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|[3.0,39.0,1.0,5.0...|
|       1.0|       0|[3.0,14.0,0.0,0.0...|
|       0.0|       0|[3.0,2.0,4.0,1.0,...|
|       0.0|       1|(10,[0,1,4,5,6],[...|
|       0.0|       1|(10,[0,1,4,5,6],[...|
|       0.0|       0|[1.0,19.0,3.0,2.0...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[3.0,21.845637583...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[3.0,14.0,1.0,0.0...|
|       1.0|       1|[3.0,21.845637583...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       0.0|       0|[3.0,16.0,5.0,2.0...|
|       0.0|       0|(10,[0,1,4,5,6],[...|
|       1.0|       1|[2.0,0.83,0.0,2.0...|
|       1.0|       1|[1.0,23.0,3.0,2.0...|
+----------

## Référence

https://spark.apache.org/docs/latest/ml-classification-regression.html